# 推理服务与量化补充线 · 第 7/8 课：Weight-only、W8A8、KV 量化与方法选择

> 状态：**未开始**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：用模型/KV 账本判断量化优先级，区分 GPTQ、AWQ、SmoothQuant 和 KV 量化解决的瓶颈。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

本课不重复量化 kernel；重点是算法类别、服务瓶颈和质量评估之间的选择。

前置：train 第 1～5 课、CUDA/Triton 基础、Transformer attention。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

Weight-only 降权重带宽/容量，适合 decode；W8A8/FP8 同时压缩激活并使用低精度 GEMM；GPTQ 逐层近似误差，AWQ 保护显著权重通道，SmoothQuant 将 activation outlier 难度平滑到权重；KV 量化独立影响长上下文容量。

### 数据与控制如何流动

先分解权重、KV、workspace 和激活的显存/带宽占比，再按目标硬件可用 kernel 选候选格式；最后以同一请求分布同时比较质量、容量、TTFT、ITL 与成本。

### 正确性条件与常见误区

格式名不等于硬件加速。必须核对目标 GPU/kernel、group size、校准和模型结构；质量验证要覆盖 perplexity、任务正确率、长上下文和推理稳定性。

### 性能、成本与工程取舍

INT4 权重理论容量减半于 INT8，但 scale/zero/packing 和反量化有开销；KV 量化提升并发却会在每个 decode step 被读取，可能直接影响 attention 输出。

## 具体演示

80GB GPU 中权重占 40GB、运行时 8GB，剩 32GB 给每请求 1GB KV：并发约 32。若权重量化到 20GB，KV 空间 52GB，并发约 52；若再把 KV 减半，约 104（均未计碎片/水位）。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐保留运行时和水位后的最大并发账本。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def max_concurrency(total_bytes, weight_bytes, runtime_bytes,
                    kv_bytes_per_request, reserve_fraction=0.1):
    if kv_bytes_per_request <= 0 or not 0 <= reserve_fraction < 1:
        raise ValueError("invalid capacity model")
    usable = int(total_bytes * (1 - reserve_fraction)) - weight_bytes - runtime_bytes
    # TODO：容量不足时返回 0，否则向下取整。
    return ______

GB = 10**9
assert max_concurrency(80*GB, 40*GB, 8*GB, 1*GB, 0.1) == 24
assert max_concurrency(80*GB, 70*GB, 8*GB, 1*GB, 0.1) == 0


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

decode 权重带宽受限时，为何 weight-only INT4 可能有收益，而 prefill 不一定同等受益？

**你的答案：**


### Q2

AWQ/GPTQ 模型文件更小，为什么服务显存不一定按相同比例下降？

**你的答案：**


### Q3

何时优先 KV 量化而不是继续压权重？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考资料

- [torchao quantized inference](https://docs.pytorch.org/ao/stable/workflows/inference.html)
- [AWQ](https://arxiv.org/abs/2306.00978)
- [GPTQ](https://arxiv.org/abs/2210.17323)
- [SmoothQuant](https://arxiv.org/abs/2211.10438)
- [vLLM serving documentation](https://docs.vllm.ai/en/latest/cli/serve/)

API 与平台能力会演进；部署前应按目标版本重新核对。